[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [2]:
import torch
import math

In [14]:
def apply_rope(q, k):
  # 1. Compute position angles
  # 2. Split into even/odd pairs
  # 3. Apply rotation

  B, S, D = q.shape

  pos = torch.arange(S, device=q.device).unsqueeze(1).float()
  dim = torch.arange(0, D, 2, device=q.device).float()
  freq = 1.0 / (10000)**((2 * dim) / D)
  theta = pos * freq # [S, D // 2]
  cos_theta = torch.cos(theta)
  sin_theta = torch.sin(theta)

  def rotate(x):
    x1 = x[..., 0::2] # Even indices
    x2 = x[..., 1::2] # Odd indices

    # Apply the rotation: [x0*cosθ - x1*sinθ, x0*sinθ + x1*cosθ]
    rotated_x1 = x1 * cos_theta - x2 * sin_theta
    rotated_x2 = x1 * sin_theta + x2 * cos_theta # Corrected this line

    return torch.stack([rotated_x1, rotated_x2], dim=-1).flatten(-2)

  return rotate(q), rotate(k)

In [15]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (2.9ms)
  ✅ [2/4] Preserves norm (3.5ms)
  ✅ [3/4] Relative position property (2.3ms)
  ✅ [4/4] Gradient flow (1.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (9.9ms total)
  Progress saved. Run status() to see your dashboard.

